# One-move blunder prediction with meta features


## Production summary

This notebook predicts whether the mover will blunder on their next own move. It uses the current causal position/history features plus game-level meta features, evaluates weighted and unweighted boosted-tree candidates on the natural validation distribution, optimizes only average precision, caches fitted workflow state to avoid repeating hyperparameter optimization, and exports figures, model artifacts, and plot source data.


In [ ]:
# imports
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd

try:
  from tqdm.auto import tqdm
except ImportError:
  def tqdm(iterable, **kwargs):
    return iterable

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
  ConfusionMatrixDisplay,
  PrecisionRecallDisplay,
  RocCurveDisplay,
  average_precision_score,
  balanced_accuracy_score,
  classification_report,
  confusion_matrix,
  f1_score,
  matthews_corrcoef,
  precision_recall_curve,
  precision_score,
  recall_score,
  roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

# Reproducible presentation defaults.
plt.rcParams.update({
  "figure.dpi": 120,
  "savefig.dpi": 300,
  "axes.spines.top": False,
  "axes.spines.right": False,
  "axes.titleweight": "bold",
  "axes.labelsize": 11,
  "axes.titlesize": 12,
  "legend.frameon": False,
  "font.size": 10,
})

RUN_NAME = "blunder-1-move-meta-features"

MAX_ABS_EVAL_PAWNS = 50.0
MISTAKE_PAWN_LOSS = 1.0
BLUNDER_PAWN_LOSS = 2.0
HORIZON_OWN_MOVES = 1
RANDOM_STATE = 42


## 1. Import data

In [ ]:
def find_project_root(start=None):
  if start is None:
    start = Path.cwd()

  start = Path(start).resolve()

  for path in [start, *start.parents]:
    has_pyproject = (path / "pyproject.toml").exists()
    has_data = (path / "data").exists()

    if has_pyproject and has_data:
      return path

  raise FileNotFoundError(
    "Could not find the project root. Run this notebook from "
    "the project or one of its subdirectories."
  )


ROOT = find_project_root()
DATA_DIR = (
  ROOT
  / "data"
  / "processed"
  / "lichess-2017-05-eval-all"
)

GAMES_PATH = DATA_DIR / "games.parquet"
PLIES_PATH = DATA_DIR / "plies.parquet"
FEATURES_PATH = DATA_DIR / "features.parquet"
DICT_PATH = DATA_DIR / "feature_dictionary.csv"

for path in [
  GAMES_PATH,
  PLIES_PATH,
  FEATURES_PATH,
  DICT_PATH,
]:
  print(f"{path.name:24s} exists={path.exists()}")

FIGURES_DIR = ROOT / "figures" / RUN_NAME
MODELS_DIR = ROOT / "models" / RUN_NAME
PLOT_DATA_DIR = ROOT / "plot-data" / RUN_NAME

for directory in [FIGURES_DIR, MODELS_DIR, PLOT_DATA_DIR]:
  directory.mkdir(parents=True, exist_ok=True)


def save_figure(fig, stem):
  png_path = FIGURES_DIR / f"{stem}.png"
  pdf_path = FIGURES_DIR / f"{stem}.pdf"
  fig.savefig(png_path, bbox_inches="tight", facecolor="white")
  fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
  print(f"saved: {png_path}")
  print(f"saved: {pdf_path}")


def save_plot_data(df, stem):
  path = PLOT_DATA_DIR / f"{stem}.csv"
  df.to_csv(path, index=False)
  print(f"saved: {path}")
  return path


In [ ]:
# inspect
def require_processed_files(paths, build_hint):
  missing = [path for path in paths if not path.exists()]
  if missing:
    missing_text = "\n".join(f"  - {path}" for path in missing)
    raise FileNotFoundError(
      "Missing processed data files:\n"
      f"{missing_text}\n\n"
      "These parquet files are generated artifacts and are not downloaded automatically.\n"
      f"Regenerate them with:\n{build_hint}"
    )


require_processed_files(
  [GAMES_PATH, PLIES_PATH, FEATURES_PATH],
  """cd blunder
uv run python process_data.py \
  --input data/raw/lichess_db_standard_rated_2017-05.pgn.zst \
  --output-dir data/processed/lichess-2017-05-eval-all \
  --batch-size 1000 \
  --format parquet""",
)

games = pd.read_parquet(GAMES_PATH)
plies = pd.read_parquet(PLIES_PATH)
features = pd.read_parquet(FEATURES_PATH)

if DICT_PATH.exists():
  feature_dict = pd.read_csv(DICT_PATH)
else:
  feature_dict = pd.DataFrame()

print("games:", games.shape)
print("plies:", plies.shape)
print("features:", features.shape)
print("feature_dict:", feature_dict.shape)


## 2. Detect the essential columns

This notebook starts from the dataframes already loaded above. The goal here is
not to create modelling features yet; it is to bind the raw table columns to the
semantic roles the rest of the blunder-prediction pipeline needs.

Ply parity is treated as authoritative for the mover:

- odd ply: White just moved;
- even ply: Black just moved.

An existing turn/side/color column is still detected, but only as a diagnostic,
because such columns may mean either "player who moved" or "player to move next"
depending on the export.


In [ ]:
REQUIRED_DATAFRAMES = ["games", "plies"]


def missing_dataframe(name):
  if name not in globals():
    return True
  return not isinstance(globals()[name], pd.DataFrame)


missing_dataframes = [
  name
  for name in REQUIRED_DATAFRAMES
  if missing_dataframe(name)
]

if missing_dataframes:
  raise NameError(
    "Run the import-data cells first. Missing dataframe(s): "
    + ", ".join(missing_dataframes)
  )

if "features" not in globals() or not isinstance(features, pd.DataFrame):
  features = pd.DataFrame()

# Manual escape hatch. Fill this only if auto-detection picks the wrong column.
# Keys are semantic roles used below; values must be actual columns in that table.
COLUMN_OVERRIDES = {
  "plies": {
    # "game_id": "game_index",
    # "ply": "ply",
    # "eval_pawns": "eval_pawns",
    # "turn": "side",
  },
  "games": {
    # "game_id": "game_index",
  },
  "features": {
    # "game_id": "game_index",
  },
}


def pick_column(
  df,
  candidates,
  *,
  table_name,
  role,
  required=True,
):
  override = COLUMN_OVERRIDES.get(table_name, {}).get(role)

  if override is not None:
    if override in df.columns:
      return override

    raise KeyError(
      f"COLUMN_OVERRIDES[{table_name!r}][{role!r}]={override!r} "
      f"is not present in {table_name}.columns"
    )

  for name in candidates:
    if name in df.columns:
      return name

  if required:
    tried = ", ".join(candidates)
    raise KeyError(
      f"Could not find {role!r} in {table_name}. Tried: {tried}"
    )

  return None


def pick_contains(
  df,
  include,
  *,
  table_name,
  role,
  exclude=None,
  required=True,
):
  if exclude is None:
    exclude = []

  include = [x.lower() for x in include]
  exclude = [x.lower() for x in exclude]

  for col in df.columns:
    low = col.lower()
    has_all = all(x in low for x in include)
    has_excluded = any(x in low for x in exclude)

    if has_all and not has_excluded:
      return col

  if required:
    raise KeyError(
      f"Could not find {role!r} in {table_name} containing {include}"
    )

  return None


GAME_ID_CANDIDATES = ["game_id", "game_index", "game_idx", "id"]
PLY_CANDIDATES = ["ply", "ply_index", "ply_number", "move_ply"]
EVAL_CANDIDATES = [
  "eval_pawns",
  "eval",
  "score",
  "engine_eval",
  "stockfish_eval",
  "stockfish_eval_pawns",
]
TURN_CANDIDATES = ["turn", "side", "color", "player_color", "mover"]

plies_game_id_col = pick_column(
  plies,
  GAME_ID_CANDIDATES,
  table_name="plies",
  role="game_id",
)

ply_col = pick_column(
  plies,
  PLY_CANDIDATES,
  table_name="plies",
  role="ply",
)

eval_col = pick_column(
  plies,
  EVAL_CANDIDATES,
  table_name="plies",
  role="eval_pawns",
  required=False,
)

if eval_col is None:
  eval_col = pick_contains(
    plies,
    ["eval"],
    table_name="plies",
    role="eval_pawns",
    exclude=["mate", "raw", "comment"],
  )

turn_col = pick_column(
  plies,
  TURN_CANDIDATES,
  table_name="plies",
  role="turn",
  required=False,
)

games_game_id_col = pick_column(
  games,
  GAME_ID_CANDIDATES,
  table_name="games",
  role="game_id",
  required=False,
)

features_game_id_col = None
if not features.empty:
  features_game_id_col = pick_column(
    features,
    GAME_ID_CANDIDATES,
    table_name="features",
    role="game_id",
    required=False,
  )

# Backwards-compatible aliases for code adapted from earlier notebooks.
game_id_col = plies_game_id_col

print("plies_game_id_col:   ", plies_game_id_col)
print("ply_col:             ", ply_col)
print("eval_col:            ", eval_col)
print("turn_col diagnostic: ", turn_col)
print("games_game_id_col:   ", games_game_id_col)
print("features_game_id_col:", features_game_id_col)


## 3. Sample complete games before feature construction

For development, sample by game id before constructing move-level targets and
features. This keeps every retained game complete, which matters because the
one-move label depends on later plies from the same game.

Set `N_GAMES_SAMPLE = None` to use all games.


In [ ]:
RANDOM_STATE = 42
N_GAMES_SAMPLE = 100_000
RANDOM_GAME_SAMPLE = True


def sample_complete_games(
  plies_df,
  *,
  n_games,
  random_sample=True,
  seed=RANDOM_STATE,
):
  if n_games is None:
    sampled_plies = plies_df.copy()
  else:
    game_ids = plies_df[plies_game_id_col].drop_duplicates()

    if n_games >= len(game_ids):
      selected_game_ids = game_ids
    elif random_sample:
      selected_game_ids = game_ids.sample(
        n=n_games,
        random_state=seed,
      )
    else:
      selected_game_ids = game_ids.head(n_games)

    sampled_plies = plies_df[
      plies_df[plies_game_id_col].isin(selected_game_ids)
    ].copy()

  return sampled_plies.sort_values(
    [plies_game_id_col, ply_col]
  ).reset_index(drop=True)


plies = sample_complete_games(
  plies,
  n_games=N_GAMES_SAMPLE,
  random_sample=RANDOM_GAME_SAMPLE,
)

sampled_game_ids = set(plies[plies_game_id_col].unique())

if games_game_id_col is not None:
  games = games[
    games[games_game_id_col].isin(sampled_game_ids)
  ].copy()

if features_game_id_col is not None:
  features = features[
    features[features_game_id_col].isin(sampled_game_ids)
  ].copy()

print("sampled games:", plies[plies_game_id_col].nunique())
print("sampled plies:", len(plies))
print("games rows:", len(games))
print("features rows:", len(features))


## 4. Construct move loss column
Be careful about alignment here.
* `evaluation before move = pervious row's evaluation`
* `evaluation after move = current row's evaluation`

In [ ]:
def add_move_columns(df):

  # Prepare the dataframe so that later calculations
  # can happen in the correct move order without modifying
  # the original input. (safety feature)
  df = df.copy()
  df = df.sort_values(
    [game_id_col, ply_col]
  ).reset_index(drop=True)

  # Extracting relevant features into new dataframe.
  df["ply_number"] = pd.to_numeric(
    df[ply_col],
    errors="coerce"  # NaN for errors
  )

  df["eval_pawns"] = pd.to_numeric(
    df[eval_col],
    errors="coerce",
  ).clip(
    -MAX_ABS_EVAL_PAWNS,
    MAX_ABS_EVAL_PAWNS
  )

  df["mover"] = np.where(
    df["ply_number"] % 2 == 1,
    "white",
    "black",
  )

  df["fullmove_number"] = (
    (df["ply_number"] + 1) // 2
  ).astype("Int64")

  group = df.groupby(game_id_col, sort=False)
  df["eval_before_pawns"] = group["eval_pawns"].shift(1)
  df["eval_delta_pawns"] = (
    df["eval_pawns"] - df["eval_before_pawns"]
  )

  white_loss = -df["eval_delta_pawns"]
  black_loss = df["eval_delta_pawns"]

  df["pawn_loss"] = np.where(
    df["mover"].eq("white"),
    white_loss,
    black_loss,
  )
  df["pawn_loss"] = df["pawn_loss"].clip(lower=0)

  df["is_mistake"] = df["pawn_loss"].ge(
    MISTAKE_PAWN_LOSS
  ).astype("Int64")

  df["is_blunder"] = df["pawn_loss"].ge(
    BLUNDER_PAWN_LOSS
  ).astype("Int64")

  missing_pair = (
    df["eval_before_pawns"].isna()
    | df["eval_pawns"].isna()
  )
  df.loc[missing_pair, "pawn_loss"] = np.nan
  df.loc[missing_pair, "is_mistake"] = pd.NA
  df.loc[missing_pair, "is_blunder"] = pd.NA

  return df


plies_ml = add_move_columns(plies)


In [ ]:
# Safety function supplied by the chat to deal with
# missing/corrupted values.
def normalize_turn_value(value):
  if pd.isna(value):
    return np.nan

  value = str(value).strip().lower()

  if value in {"w", "white", "1", "true"}:
    return "white"

  if value in {"b", "black", "0", "false"}:
    return "black"

  return value


if turn_col is not None:
  supplied_turn = plies_ml[turn_col].map(normalize_turn_value)
  direct_match = supplied_turn.eq(plies_ml["mover"]).mean()
  opposite_match = supplied_turn.ne(plies_ml["mover"]).mean()

  print("Turn-column diagnostic")
  print("matches mover from parity: ", direct_match)
  print("differs from parity mover:  ", opposite_match)

show_cols = [
  game_id_col,
  ply_col,
  "mover",
  "eval_before_pawns",
  "eval_pawns",
  "eval_delta_pawns",
  "pawn_loss",
  "is_blunder",
]

display(plies_ml[show_cols].head(100))


In [ ]:
# Sanity check: what fraction of moves are mistakes/blunders
print("Move-level event rates")
display(
  plies_ml[["is_mistake", "is_blunder"]]
  .astype(float)
  .mean()
  .to_frame("fraction")
)

# And also the largest measured pawn losses
# remember we clip at MAX_ABS_EVAL_PAWNS
print("Largest measured pawn losses")
display(
  plies_ml[show_cols]
  .sort_values("pawn_loss", ascending=False)
  .head(20)
)


## 5. Build the one-move-ahead target


In [ ]:
def add_future_target(df, horizon):
  if horizon != 1:
    raise ValueError(
      "This notebook is for a one-own-move prediction horizon."
    )

  df = df.copy()
  df = df.sort_values(
    [game_id_col, "mover", ply_col]
  )

  keys = [game_id_col, "mover"]
  next_blunder = (
    df.groupby(keys, sort=False)["is_blunder"]
    .shift(-1)
  )

  df["future_blunder_count"] = next_blunder
  df["will_blunder_next_move"] = next_blunder.astype(float)

  return df.sort_values(
    [game_id_col, ply_col]
  ).reset_index(drop=True)


plies_ml = add_future_target(
  plies_ml,
  HORIZON_OWN_MOVES,
)

print("One-move-ahead target counts, including unknown rows")
display(
  plies_ml["will_blunder_next_move"]
  .value_counts(dropna=False)
)

target_ordered = plies_ml.sort_values(
  [game_id_col, "mover", ply_col]
).copy()
next_label = (
  target_ordered
  .groupby([game_id_col, "mover"], sort=False)["is_blunder"]
  .shift(-1)
)
future_moves_available = target_ordered.groupby(
  [game_id_col, "mover"],
  sort=False,
).cumcount(ascending=False)
missing_target = target_ordered["will_blunder_next_move"].isna()
end_of_game_censored = (
  missing_target
  & future_moves_available.eq(0)
)
missing_next_label = (
  missing_target
  & future_moves_available.ge(1)
)

print("Why the one-move target is missing")
display(
  pd.Series({
    "no_later_own_move": int(end_of_game_censored.sum()),
    "next_move_exists_but_label_is_missing": int(
      missing_next_label.sum()
    ),
    "missing_targets_total": int(missing_target.sum()),
  }).to_frame("rows")
)

print("Observed one-move-ahead prevalence")
print(target_ordered["will_blunder_next_move"].mean())


### Concrete target-window example


In [ ]:
target_example_cols = [
  game_id_col,
  ply_col,
  "mover",
  "pawn_loss",
  "is_blunder",
  "future_blunder_count",
  "will_blunder_next_move",
]

print("Rows 20-29 from plies_ml")
display(plies_ml.loc[20:29, target_example_cols])


def show_future_window(row_index):
  row = plies_ml.loc[row_index]
  future_window = (
    plies_ml[
      plies_ml[game_id_col].eq(row[game_id_col])
      & plies_ml["mover"].eq(row["mover"])
      & plies_ml[ply_col].gt(row[ply_col])
    ]
    .sort_values(ply_col)
    .head(HORIZON_OWN_MOVES)
  )

  print(
    f"row {row_index}: game={row[game_id_col]}, "
    f"ply={row[ply_col]}, mover={row['mover']}"
  )
  print(
    f"future rows found: {len(future_window)} / "
    f"{HORIZON_OWN_MOVES}; "
    f"target={row['will_blunder_next_move']}; "
    f"future_blunder_count={row['future_blunder_count']}"
  )
  display(future_window[target_example_cols])


show_future_window(20)
show_future_window(24)


Rows without an observed next own move, or whose next own move lacks an
`is_blunder` label, have an unknown target. Those rows are excluded from
modelling rather than silently treated as non-blunders.


## 6. Construct causal features plus game metadata

The base features are known immediately after the current move. The meta-feature workflow additionally uses information known before the game starts: ratings, rating gaps, time-control fields, and event flags.

In [ ]:
def classify_time_control(base_seconds):
  if pd.isna(base_seconds):
    return "unknown"

  base_seconds = float(base_seconds)

  if base_seconds < 180:
    return "bullet"

  if base_seconds < 480:
    return "blitz"

  if base_seconds < 1500:
    return "rapid"

  return "classical"


def build_game_meta_features(games_df):
  meta_cols = [
    game_id_col,
    "white_elo",
    "black_elo",
    "avg_elo",
    "elo_diff_white_minus_black",
    "time_base_seconds",
    "time_increment_seconds",
    "event",
    "time_control",
  ]

  meta = games_df[meta_cols].copy()

  meta["white_elo"] = pd.to_numeric(meta["white_elo"], errors="coerce")
  meta["black_elo"] = pd.to_numeric(meta["black_elo"], errors="coerce")
  meta["avg_elo"] = pd.to_numeric(meta["avg_elo"], errors="coerce")
  meta["elo_diff_white_minus_black"] = pd.to_numeric(
    meta["elo_diff_white_minus_black"],
    errors="coerce",
  )
  meta["time_base_seconds"] = pd.to_numeric(
    meta["time_base_seconds"],
    errors="coerce",
  )
  meta["time_increment_seconds"] = pd.to_numeric(
    meta["time_increment_seconds"],
    errors="coerce",
  )

  meta["log_time_base_seconds"] = np.log1p(meta["time_base_seconds"])
  meta["log_time_increment_seconds"] = np.log1p(
    meta["time_increment_seconds"]
  )
  meta["has_increment"] = (
    meta["time_increment_seconds"].gt(0).astype(float)
  )
  meta["time_control_category"] = meta["time_base_seconds"].map(
    classify_time_control
  )

  event_lower = meta["event"].fillna("").str.lower()
  meta["event_is_rated"] = event_lower.str.contains("rated").astype(float)
  meta["event_is_tournament"] = event_lower.str.contains(
    "tournament"
  ).astype(float)

  return meta.drop(columns=["event", "time_control"])


def add_causal_features(df):
  df = df.copy()
  df = df.sort_values(
    [game_id_col, ply_col]
  ).reset_index(drop=True)

  game_meta = build_game_meta_features(games)
  df = df.merge(
    game_meta,
    on=game_id_col,
    how="left",
    validate="many_to_one",
  )

  game_group = df.groupby(game_id_col, sort=False)
  player_keys = [game_id_col, "mover"]
  player_group = df.groupby(player_keys, sort=False)

  mover_sign = np.where(df["mover"].eq("white"), 1.0, -1.0)
  df["mover_is_white"] = df["mover"].eq("white").astype(int)

  # Game metadata from the mover's perspective. These are known before the game
  # starts, but side-relative versions are easier for the model to use.
  df["mover_elo"] = np.where(
    df["mover"].eq("white"),
    df["white_elo"],
    df["black_elo"],
  )
  df["opponent_elo"] = np.where(
    df["mover"].eq("white"),
    df["black_elo"],
    df["white_elo"],
  )
  df["elo_diff_for_mover"] = df["mover_elo"] - df["opponent_elo"]
  df["abs_elo_diff"] = df["elo_diff_for_mover"].abs()

  # Position/eval immediately after the current move, from the mover's
  # perspective. Positive means the mover stands better.
  df["eval_for_mover"] = df["eval_pawns"] * mover_sign
  df["eval_before_for_mover"] = df["eval_before_pawns"] * mover_sign
  df["abs_eval_pawns"] = df["eval_pawns"].abs()

  # Current move quality. This is causal because the prediction is made after
  # the move has been played and evaluated.
  df["current_pawn_loss"] = df["pawn_loss"]
  df["current_is_mistake"] = df["is_mistake"].astype(float)
  df["current_is_blunder"] = df["is_blunder"].astype(float)
  df["current_eval_swing"] = df["eval_delta_pawns"]
  df["current_eval_swing_for_mover"] = df["eval_delta_pawns"] * mover_sign
  df["abs_current_eval_swing"] = df["current_eval_swing"].abs()

  # Board-derived phase and material signals already exist in the plies table.
  # They describe the current board, unlike full-game summaries in features.
  df["phase_progress_current"] = df["phase_progress"]
  df["opening_weight_current"] = df["opening_like_weight"]
  df["middlegame_weight_current"] = df["middlegame_like_weight"]
  df["endgame_weight_current"] = df["endgame_like_weight"]
  df["material_imbalance_for_mover"] = (
    df["material_imbalance_white"] * mover_sign
  )
  df["non_pawn_imbalance_for_mover"] = (
    df["non_pawn_imbalance_white"] * mover_sign
  )
  df["total_material_current"] = df["total_material"]
  df["total_non_pawn_material_current"] = df["total_non_pawn_material"]
  df["both_queens_present_current"] = (
    df["both_queens_present"].astype(float)
  )
  df["no_queens_present_current"] = df["no_queens_present"].astype(float)

  # Clock pressure after the current move. Low clock is an obvious candidate for
  # future blunders and is available at prediction time.
  df["clock_seconds_current"] = df["clock_seconds"]
  df["log_clock_seconds_current"] = np.log1p(df["clock_seconds"])
  df["previous_own_clock_seconds"] = player_group[
    "clock_seconds"
  ].shift(1)
  df["own_clock_change"] = (
    df["clock_seconds"] - df["previous_own_clock_seconds"]
  )

  # Historical own-move quality. Shift first so these summaries never include
  # the current row unless explicitly named current_* above.
  previous_loss = player_group["pawn_loss"].shift(1)
  previous_mistake = player_group["is_mistake"].shift(1)
  previous_blunder = player_group["is_blunder"].shift(1)

  hist_keys = [df[game_id_col], df["mover"]]
  df["previous_own_pawn_loss"] = previous_loss
  df["previous_own_mistakes"] = (
    previous_mistake.fillna(0).groupby(hist_keys, sort=False).cumsum()
  )
  df["previous_own_blunders"] = (
    previous_blunder.fillna(0).groupby(hist_keys, sort=False).cumsum()
  )

  for window in [3, 5, 10]:
    rolled = (
      previous_loss
      .groupby(hist_keys, sort=False)
      .rolling(window, min_periods=1)
      .agg(["mean", "max", "std"])
      .reset_index(level=[0, 1], drop=True)
    )

    df[f"previous_loss_mean_{window}"] = rolled["mean"]
    df[f"previous_loss_max_{window}"] = rolled["max"]
    df[f"previous_loss_std_{window}"] = rolled["std"]

  # A move-number feature can still be useful as tempo/elapsed-game information,
  # but it is no longer asked to stand in for chess phase.
  df["fullmove_number_log"] = np.log1p(
    df["fullmove_number"].astype(float)
  )

  return df


plies_ml = add_causal_features(plies_ml)


## 7. Define the all-features model table


In [ ]:
BASE_CANDIDATE_FEATURES = [
  "mover_is_white",
  "fullmove_number",
  "fullmove_number_log",
  "eval_for_mover",
  "eval_before_for_mover",
  "abs_eval_pawns",
  "current_pawn_loss",
  "current_is_mistake",
  "current_is_blunder",
  "current_eval_swing_for_mover",
  "abs_current_eval_swing",
  "phase_progress_current",
  "opening_weight_current",
  "middlegame_weight_current",
  "endgame_weight_current",
  "material_imbalance_for_mover",
  "non_pawn_imbalance_for_mover",
  "total_material_current",
  "total_non_pawn_material_current",
  "both_queens_present_current",
  "no_queens_present_current",
  "clock_seconds_current",
  "log_clock_seconds_current",
  "previous_own_clock_seconds",
  "own_clock_change",
  "previous_own_pawn_loss",
  "previous_own_mistakes",
  "previous_own_blunders",
  "previous_loss_mean_3",
  "previous_loss_max_3",
  "previous_loss_std_3",
  "previous_loss_mean_5",
  "previous_loss_max_5",
  "previous_loss_std_5",
  "previous_loss_mean_10",
  "previous_loss_max_10",
  "previous_loss_std_10",
]

META_FEATURES = [
  "mover_elo",
  "opponent_elo",
  "avg_elo",
  "elo_diff_for_mover",
  "abs_elo_diff",
  "time_base_seconds",
  "time_increment_seconds",
  "log_time_base_seconds",
  "log_time_increment_seconds",
  "has_increment",
  "time_control_category",
  "event_is_rated",
  "event_is_tournament",
]

base_feature_cols = [
  col
  for col in BASE_CANDIDATE_FEATURES
  if col in plies_ml.columns
]
meta_feature_cols = [
  col
  for col in META_FEATURES
  if col in plies_ml.columns
]
all_feature_cols = list(dict.fromkeys([
  *base_feature_cols,
  *meta_feature_cols,
]))

print(f"Base causal features: {len(base_feature_cols)}")
print(f"Meta features: {len(meta_feature_cols)}")
print(f"All model features: {len(all_feature_cols)}")
print("Meta features included:")
for col in meta_feature_cols:
  print(col)


## 8. Create the one-move modelling table


In [ ]:
model_df = plies_ml[
  plies_ml["will_blunder_next_move"].notna()
].copy()
X_all = model_df[all_feature_cols]
y = model_df["will_blunder_next_move"].astype(int)
groups = model_df[game_id_col]

print("model rows:", len(model_df))
print("unique games:", groups.nunique())
print("natural positive prevalence:", y.mean())
display(y.value_counts().sort_index().to_frame("rows"))

feature_missingness = (
  X_all.isna()
  .mean()
  .sort_values(ascending=False)
  .to_frame("missing_fraction")
)
display(feature_missingness.head(25))


## 9. Split by complete games

The train/validation/test split must happen by game, not by row. Rows from the
same chess game are highly related: neighbouring plies share position, clock,
players, opening, and future game context. If one game appeared in both train
and validation/test, the evaluation would be too optimistic.

The split below first creates a one-row-per-game table and marks whether each
game has at least one positive labelled row. It then stratifies on that game
level flag so all splits get a reasonable share of games containing positives,
while still keeping each game entirely in one split.

Validation and test are left at their natural row-level prevalence. They are not
balanced here; balancing/downsampling, if used, should happen only inside the
training split.


In [ ]:
def game_train_valid_calibration_test_split(
  X_all,
  y,
  groups,
  seed=RANDOM_STATE,
):
  """Split complete games into 70/10/10/10 row-independent roles."""
  game_table = (
    pd.DataFrame({
      "game_id": groups,
      "has_positive": y,
    })
    .groupby("game_id", as_index=False)["has_positive"]
    .max()
  )

  train_games, remainder_games = train_test_split(
    game_table,
    train_size=0.70,
    random_state=seed,
    stratify=game_table["has_positive"],
  )

  valid_games, cal_test_games = train_test_split(
    remainder_games,
    train_size=1 / 3,
    random_state=seed,
    stratify=remainder_games["has_positive"],
  )

  calibration_games, test_games = train_test_split(
    cal_test_games,
    train_size=0.50,
    random_state=seed,
    stratify=cal_test_games["has_positive"],
  )

  game_ids = {
    "train": set(train_games["game_id"]),
    "valid": set(valid_games["game_id"]),
    "calibration": set(calibration_games["game_id"]),
    "test": set(test_games["game_id"]),
  }

  split_names = list(game_ids)
  for i, left in enumerate(split_names):
    for right in split_names[i + 1:]:
      assert game_ids[left].isdisjoint(game_ids[right])

  output = []
  for split_name in split_names:
    mask = groups.isin(game_ids[split_name])
    output.extend([
      X_all.loc[mask].copy(),
      y.loc[mask].copy(),
      groups.loc[mask].copy(),
    ])

  return tuple(output)


(
  X_train,
  y_train,
  g_train,
  X_valid,
  y_valid,
  g_valid,
  X_calibration,
  y_calibration,
  g_calibration,
  X_test,
  y_test,
  g_test,
) = game_train_valid_calibration_test_split(X_all, y, groups)

split_summary = pd.DataFrame([
  {
    "split": name,
    "rows": len(split_y),
    "games": split_g.nunique(),
    "positive_prevalence": split_y.mean(),
  }
  for name, split_y, split_g in [
    ("train", y_train, g_train),
    ("valid", y_valid, g_valid),
    ("calibration", y_calibration, g_calibration),
    ("test", y_test, g_test),
  ]
])

display(split_summary)
print("All complete-game split overlap checks passed.")


## 10. Shared modelling helpers


In [ ]:
def make_balanced_sample_weights(y_values):
  n_pos = y_values.eq(1).sum()
  n_neg = y_values.eq(0).sum()
  if n_pos == 0 or n_neg == 0:
    raise ValueError("Both classes are required for weighting.")
  pos_weight = len(y_values) / (2 * n_pos)
  neg_weight = len(y_values) / (2 * n_neg)
  return np.where(y_values.to_numpy() == 1, pos_weight, neg_weight)


sample_weight_train_balanced = make_balanced_sample_weights(y_train)

def get_score(model, X_eval):
  if hasattr(model, "predict_proba"):
    probabilities = model.predict_proba(X_eval)
    positive_index = list(model.classes_).index(1)
    return probabilities[:, positive_index]
  score = model.decision_function(X_eval)
  return 1 / (1 + np.exp(-score))


def ranking_metrics(name, model, X_eval, y_eval):
  score = get_score(model, X_eval)
  prevalence = y_eval.mean()
  average_precision = average_precision_score(y_eval, score)
  return {
    "model": name,
    "roc_auc": roc_auc_score(y_eval, score),
    "average_precision": average_precision,
    "prevalence": prevalence,
    "average_precision_lift_over_random": average_precision / prevalence,
  }


def threshold_metric_table(y_true, score):
  y_array = np.asarray(y_true, dtype=np.int8)
  score_array = np.asarray(score, dtype=float)

  if len(y_array) != len(score_array):
    raise ValueError("y_true and score must have the same length.")

  if len(y_array) == 0:
    return pd.DataFrame(columns=[
      "threshold",
      "precision",
      "recall",
      "f1",
      "predicted_positive_rate",
      "mcc",
    ])

  order = np.argsort(score_array)[::-1]
  sorted_score = score_array[order]
  sorted_y = y_array[order]
  distinct_ends = np.r_[
    np.flatnonzero(np.diff(sorted_score)),
    len(sorted_score) - 1,
  ]

  threshold_desc = sorted_score[distinct_ends]
  predicted_positive = distinct_ends + 1
  true_positive = np.cumsum(sorted_y)[distinct_ends].astype(float)
  false_positive = predicted_positive - true_positive

  total_positive = float(sorted_y.sum())
  total_negative = float(len(sorted_y) - total_positive)
  false_negative = total_positive - true_positive
  true_negative = total_negative - false_positive

  precision = np.divide(
    true_positive,
    predicted_positive,
    out=np.zeros_like(true_positive, dtype=float),
    where=predicted_positive != 0,
  )
  recall = np.divide(
    true_positive,
    total_positive,
    out=np.zeros_like(true_positive, dtype=float),
    where=total_positive != 0,
  )
  f1 = np.divide(
    2 * precision * recall,
    precision + recall,
    out=np.zeros_like(precision, dtype=float),
    where=(precision + recall) != 0,
  )

  denominator = np.sqrt(
    (true_positive + false_positive)
    * (true_positive + false_negative)
    * (true_negative + false_positive)
    * (true_negative + false_negative)
  )
  mcc = np.divide(
    true_positive * true_negative - false_positive * false_negative,
    denominator,
    out=np.zeros_like(true_positive, dtype=float),
    where=denominator != 0,
  )

  table = pd.DataFrame({
    "threshold": threshold_desc,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "predicted_positive_rate": predicted_positive / len(sorted_y),
    "mcc": mcc,
  })
  return table.sort_values("threshold", ignore_index=True)


def best_threshold_row(threshold_table, metric):
  if metric == "f1":
    sort_cols = ["f1", "precision", "threshold"]
  elif metric == "mcc":
    sort_cols = ["mcc", "f1", "precision", "threshold"]
  else:
    raise ValueError(f"Unsupported threshold metric: {metric}")
  return threshold_table.sort_values(sort_cols, ascending=False).iloc[0]


def make_preprocess(feature_cols, X_train_context):
  numeric_cols = [
    col
    for col in feature_cols
    if pd.api.types.is_numeric_dtype(X_train_context[col])
  ]
  categorical_cols = [
    col for col in feature_cols
    if col not in numeric_cols
  ]
  numeric_preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
  ])
  categorical_preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
      "onehot",
      OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
      ),
    ),
  ])
  return ColumnTransformer([
    ("numeric", numeric_preprocess, numeric_cols),
    ("categorical", categorical_preprocess, categorical_cols),
  ])


def dataframe_to_numpy(X):
  if hasattr(X, "to_numpy"):
    return X.to_numpy()
  return X


def make_to_numpy():
  return FunctionTransformer(dataframe_to_numpy, validate=False)


def make_xgboost_model(feature_cols, X_train_context, model_params=None):
  params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "n_estimators": 450,
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_child_weight": 20,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_lambda": 5.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
  }
  if model_params is not None:
    params.update(model_params)
  return Pipeline([
    ("preprocess", make_preprocess(feature_cols, X_train_context)),
    ("to_numpy", make_to_numpy()),
    ("model", XGBClassifier(**params)),
  ])


def make_lightgbm_model(feature_cols, X_train_context, model_params=None):
  params = {
    "objective": "binary",
    "n_estimators": 600,
    "learning_rate": 0.04,
    "num_leaves": 63,
    "min_child_samples": 80,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_lambda": 5.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": -1,
  }
  if model_params is not None:
    params.update(model_params)
  return Pipeline([
    ("preprocess", make_preprocess(feature_cols, X_train_context)),
    ("to_numpy", make_to_numpy()),
    ("model", LGBMClassifier(**params)),
  ])


def make_candidate_models(feature_cols, X_train_context):
  return {
    "dummy_prior": {"pipeline": Pipeline([
      ("preprocess", make_preprocess(feature_cols, X_train_context)),
      ("to_numpy", make_to_numpy()),
      ("model", DummyClassifier(strategy="prior")),
    ]), "training_variant": "full_unweighted"},
    "xgboost_unweighted": {
      "pipeline": make_xgboost_model(feature_cols, X_train_context),
      "training_variant": "full_unweighted",
    },
    "xgboost_weighted": {
      "pipeline": make_xgboost_model(feature_cols, X_train_context),
      "training_variant": "full_weighted",
    },
    "lightgbm_unweighted": {
      "pipeline": make_lightgbm_model(feature_cols, X_train_context),
      "training_variant": "full_unweighted",
    },
    "lightgbm_weighted": {
      "pipeline": make_lightgbm_model(feature_cols, X_train_context),
      "training_variant": "full_weighted",
    },
  }


## 11. Train, tune, and evaluate all-features candidates


In [ ]:
N_OPTUNA_TRIALS = 30
OPTUNA_TRAIN_SAMPLE_ROWS = 500_000
OPTUNA_VALID_SAMPLE_ROWS = 200_000
WORKFLOW_CACHE_PATH = MODELS_DIR / "workflow_cache.joblib"


def model_class_label(model_name):
  if model_name.startswith("dummy"):
    return "Dummy prior"
  if model_name.startswith("xgboost_unweighted"):
    return "XGBoost unweighted"
  if model_name.startswith("xgboost_weighted"):
    return "XGBoost weighted"
  if model_name.startswith("lightgbm_unweighted"):
    return "LightGBM unweighted"
  if model_name.startswith("lightgbm_weighted"):
    return "LightGBM weighted"
  return model_name


def training_kind(model_name):
  if "weighted" in model_name and "unweighted" not in model_name:
    return "weighted"
  if "unweighted" in model_name:
    return "unweighted"
  return "baseline"


def family_for_model(model_name):
  if model_name.startswith("lightgbm"):
    return "lightgbm"
  if model_name.startswith("xgboost"):
    return "xgboost"
  raise ValueError(f"Unsupported model family: {model_name}")


def suggest_lightgbm_params(trial):
  return {
    "n_estimators": trial.suggest_int(
      "n_estimators", 300, 1200, step=100,
    ),
    "learning_rate": trial.suggest_float(
      "learning_rate", 0.015, 0.10, log=True,
    ),
    "num_leaves": trial.suggest_int(
      "num_leaves", 31, 255, log=True,
    ),
    "min_child_samples": trial.suggest_int(
      "min_child_samples", 30, 300,
    ),
    "subsample": trial.suggest_float("subsample", 0.70, 1.00),
    "colsample_bytree": trial.suggest_float(
      "colsample_bytree", 0.60, 1.00,
    ),
    "reg_alpha": trial.suggest_float(
      "reg_alpha", 1e-4, 10.0, log=True,
    ),
    "reg_lambda": trial.suggest_float(
      "reg_lambda", 1e-3, 30.0, log=True,
    ),
  }


def suggest_xgboost_params(trial):
  return {
    "n_estimators": trial.suggest_int(
      "n_estimators", 300, 1000, step=100,
    ),
    "learning_rate": trial.suggest_float(
      "learning_rate", 0.015, 0.10, log=True,
    ),
    "max_depth": trial.suggest_int("max_depth", 3, 8),
    "min_child_weight": trial.suggest_float(
      "min_child_weight", 3.0, 80.0, log=True,
    ),
    "subsample": trial.suggest_float("subsample", 0.70, 1.00),
    "colsample_bytree": trial.suggest_float(
      "colsample_bytree", 0.60, 1.00,
    ),
    "reg_alpha": trial.suggest_float(
      "reg_alpha", 1e-4, 10.0, log=True,
    ),
    "reg_lambda": trial.suggest_float(
      "reg_lambda", 1e-3, 30.0, log=True,
    ),
  }


def ensure_dummy_model(workflow):
  if "dummy_prior" in workflow["fitted_models"]:
    return
  model = Pipeline([
    ("preprocess", make_preprocess(all_feature_cols, X_train)),
    ("to_numpy", make_to_numpy()),
    ("model", DummyClassifier(strategy="prior")),
  ])
  model.fit(X_train, y_train)
  workflow["fitted_models"]["dummy_prior"] = model


def compute_operating_points(workflow):
  score = get_score(workflow["best_model"], X_calibration)
  table = threshold_metric_table(y_calibration, score)
  workflow["threshold_table"] = table
  workflow["max_f1"] = best_threshold_row(table, "f1")
  workflow["max_mcc"] = best_threshold_row(table, "mcc")

  test_score = get_score(workflow["best_model"], X_test)
  workflow["test_score"] = test_score
  workflow["predictions"] = {}
  workflow["decision_rows"] = []

  rows = [
    ("maximum calibration F1", workflow["max_f1"]),
    ("maximum calibration MCC", workflow["max_mcc"]),
  ]
  for policy, row in rows:
    threshold = float(row["threshold"])
    test_pred = (test_score >= threshold).astype(int)
    workflow["predictions"][policy] = test_pred
    test_average_precision = average_precision_score(y_test, test_score)
    workflow["decision_rows"].append({
      "model": workflow["best_model_name"],
      "threshold_policy": policy,
      "threshold": threshold,
      "calibration_precision": float(row["precision"]),
      "calibration_recall": float(row["recall"]),
      "calibration_f1": float(row["f1"]),
      "calibration_mcc": float(row["mcc"]),
      "test_roc_auc": float(roc_auc_score(y_test, test_score)),
      "test_average_precision": float(test_average_precision),
      "test_average_precision_lift_over_random": float(
        test_average_precision / y_test.mean()
      ),
      "test_balanced_accuracy": float(
        balanced_accuracy_score(y_test, test_pred)
      ),
      "test_precision": float(precision_score(
        y_test,
        test_pred,
        zero_division=0,
      )),
      "test_recall": float(recall_score(
        y_test,
        test_pred,
        zero_division=0,
      )),
      "test_f1": float(f1_score(
        y_test,
        test_pred,
        zero_division=0,
      )),
      "test_mcc": float(matthews_corrcoef(y_test, test_pred)),
      "test_predicted_positive_rate": float(test_pred.mean()),
    })


def load_cached_workflow():
  if not WORKFLOW_CACHE_PATH.exists():
    return None
  cache = joblib.load(WORKFLOW_CACHE_PATH)
  print(f"loaded: {WORKFLOW_CACHE_PATH}")
  workflow = {
    "feature_cols": all_feature_cols,
    "fitted_models": cache["fitted_models"],
    "results_df": cache["results_df"],
    "optimized_model_name": cache.get("optimized_model_name"),
    "best_model_name": cache["best_model_name"],
  }
  workflow["best_model"] = workflow["fitted_models"][
    workflow["best_model_name"]
  ]
  ensure_dummy_model(workflow)
  compute_operating_points(workflow)
  return workflow


def run_average_precision_workflow():
  cached = load_cached_workflow()
  if cached is not None:
    return cached

  training_variants = {
    "full_unweighted": {
      "X": X_train,
      "y": y_train,
      "sample_weight": None,
    },
    "full_weighted": {
      "X": X_train,
      "y": y_train,
      "sample_weight": sample_weight_train_balanced,
    },
  }
  models = make_candidate_models(all_feature_cols, X_train)

  fitted_models = {}
  results = []
  training_timings = []
  iterator = tqdm(list(models.items()), desc="candidate models")
  for name, spec in iterator:
    model = spec["pipeline"]
    variant = training_variants[spec["training_variant"]]
    fit_kwargs = {}
    if variant["sample_weight"] is not None:
      fit_kwargs["model__sample_weight"] = variant["sample_weight"]

    fit_start = perf_counter()
    model.fit(variant["X"], variant["y"], **fit_kwargs)
    fit_seconds = perf_counter() - fit_start
    fitted_models[name] = model
    results.append(ranking_metrics(name, model, X_valid, y_valid))
    training_timings.append({
      "model": name,
      "fit_seconds": fit_seconds,
    })

  results_df = pd.DataFrame(results).sort_values(
    "average_precision",
    ascending=False,
  )
  save_plot_data(results_df, "validation_baseline_metrics")
  save_plot_data(
    pd.DataFrame(training_timings),
    "baseline_training_timings",
  )

  non_dummy = results_df[~results_df["model"].str.startswith("dummy")]
  hpo_base_model_name = non_dummy.iloc[0]["model"]
  variant_name = models[hpo_base_model_name]["training_variant"]
  hpo_training_variant = training_variants[variant_name]
  hpo_family = family_for_model(hpo_base_model_name)

  if (
    OPTUNA_VALID_SAMPLE_ROWS is None
    or OPTUNA_VALID_SAMPLE_ROWS >= len(X_valid)
  ):
    X_valid_hpo = X_valid
    y_valid_hpo = y_valid
  else:
    X_valid_hpo = X_valid.sample(
      OPTUNA_VALID_SAMPLE_ROWS,
      random_state=RANDOM_STATE,
    )
    y_valid_hpo = y_valid.loc[X_valid_hpo.index]

  if (
    OPTUNA_TRAIN_SAMPLE_ROWS is None
    or OPTUNA_TRAIN_SAMPLE_ROWS >= len(hpo_training_variant["X"])
  ):
    X_train_hpo = hpo_training_variant["X"]
    y_train_hpo = hpo_training_variant["y"]
    sample_weight_hpo = hpo_training_variant["sample_weight"]
  else:
    X_train_hpo = hpo_training_variant["X"].sample(
      OPTUNA_TRAIN_SAMPLE_ROWS,
      random_state=RANDOM_STATE,
    )
    y_train_hpo = hpo_training_variant["y"].loc[X_train_hpo.index]
    if hpo_training_variant["sample_weight"] is None:
      sample_weight_hpo = None
    else:
      weight_series = pd.Series(
        hpo_training_variant["sample_weight"],
        index=hpo_training_variant["y"].index,
      )
      sample_weight_hpo = weight_series.loc[
        X_train_hpo.index
      ].to_numpy()

  def make_hpo_model(params):
    if hpo_family == "lightgbm":
      return make_lightgbm_model(all_feature_cols, X_train, params)
    return make_xgboost_model(all_feature_cols, X_train, params)

  def objective(trial):
    if hpo_family == "lightgbm":
      params = suggest_lightgbm_params(trial)
    else:
      params = suggest_xgboost_params(trial)
    model = make_hpo_model(params)
    fit_kwargs = {}
    if sample_weight_hpo is not None:
      fit_kwargs["model__sample_weight"] = sample_weight_hpo
    model.fit(X_train_hpo, y_train_hpo, **fit_kwargs)
    score = get_score(model, X_valid_hpo)
    return average_precision_score(y_valid_hpo, score)

  sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
  optuna.logging.set_verbosity(optuna.logging.WARNING)
  study = optuna.create_study(
    direction="maximize",
    study_name=f"one_move_{hpo_base_model_name}",
    sampler=sampler,
  )
  study.optimize(
    objective,
    n_trials=N_OPTUNA_TRIALS,
    show_progress_bar=True,
  )

  optimized_model_name = (
    f"{hpo_base_model_name}_optimized_average_precision"
  )
  optimized_model = make_hpo_model(study.best_params)
  fit_kwargs = {}
  if hpo_training_variant["sample_weight"] is not None:
    fit_kwargs["model__sample_weight"] = hpo_training_variant[
      "sample_weight"
    ]
  optimized_model.fit(
    hpo_training_variant["X"],
    hpo_training_variant["y"],
    **fit_kwargs,
  )
  fitted_models[optimized_model_name] = optimized_model
  optimized_metrics = ranking_metrics(
    optimized_model_name,
    optimized_model,
    X_valid,
    y_valid,
  )
  results_df = pd.concat(
    [results_df, pd.DataFrame([optimized_metrics])],
    ignore_index=True,
  ).sort_values("average_precision", ascending=False)

  workflow = {
    "feature_cols": all_feature_cols,
    "fitted_models": fitted_models,
    "results_df": results_df,
    "optimized_model_name": optimized_model_name,
    "best_model_name": results_df.iloc[0]["model"],
  }
  workflow["best_model"] = fitted_models[workflow["best_model_name"]]
  compute_operating_points(workflow)

  save_plot_data(results_df, "validation_all_metrics")
  trial_df = study.trials_dataframe().sort_values(
    "value",
    ascending=False,
  )
  save_plot_data(trial_df, "hyperparameter_trials")
  save_plot_data(
    workflow["threshold_table"],
    "calibration_threshold_curves",
  )

  cache = {
    "schema_version": 1,
    "feature_set": "all_features",
    "fitted_models": fitted_models,
    "results_df": results_df,
    "optimized_model_name": optimized_model_name,
    "best_model_name": workflow["best_model_name"],
    "feature_columns": all_feature_cols,
  }
  joblib.dump(cache, WORKFLOW_CACHE_PATH)
  print(f"saved: {WORKFLOW_CACHE_PATH}")
  return workflow


workflow = run_average_precision_workflow()
results_df = workflow["results_df"].copy()
decision_matrix_df = pd.DataFrame(workflow["decision_rows"])
save_plot_data(decision_matrix_df, "decision_matrix")
display(results_df)
display(decision_matrix_df)


## 12. Validation comparison and defensibility check


In [ ]:
from sklearn.metrics import roc_curve
from matplotlib.patches import Rectangle

model_order = [
  "Dummy prior",
  "XGBoost unweighted",
  "XGBoost weighted",
  "LightGBM unweighted",
  "LightGBM weighted",
]

comparison_df = results_df.copy()
comparison_df["model_class"] = comparison_df["model"].map(
  model_class_label
)
comparison_df["training_kind"] = comparison_df["model"].map(
  training_kind
)
comparison_df = (
  comparison_df
  .sort_values("average_precision", ascending=False)
  .groupby("model_class", as_index=False)
  .head(1)
)
comparison_df["model_class"] = pd.Categorical(
  comparison_df["model_class"],
  categories=model_order,
  ordered=True,
)
comparison_df = comparison_df.sort_values("model_class")
save_plot_data(comparison_df, "validation_comparison_model_rows")

def best_average_precision(kind):
  subset = comparison_df[comparison_df["training_kind"].eq(kind)]
  if subset.empty:
    return np.nan
  return subset["average_precision"].max()


best_weighted = best_average_precision("weighted")
best_unweighted = best_average_precision("unweighted")
weighted_is_best = bool(best_weighted >= best_unweighted)
weighted_close = bool(best_weighted >= 0.98 * best_unweighted)
weighted_defensible = weighted_is_best or weighted_close

def family_metric(family, kind):
  mask = comparison_df["model"].str.startswith(family)
  mask &= comparison_df["training_kind"].eq(kind)
  values = comparison_df.loc[mask, "average_precision"]
  if values.empty:
    return np.nan
  return values.max()


family_rows = []
for family in ["xgboost", "lightgbm"]:
  weighted_ap = family_metric(family, "weighted")
  unweighted_ap = family_metric(family, "unweighted")
  family_rows.append({
    "family": family,
    "weighted_average_precision": weighted_ap,
    "unweighted_average_precision": unweighted_ap,
    "weighted_minus_unweighted": weighted_ap - unweighted_ap,
  })
defensibility_df = pd.DataFrame(family_rows)
defensibility_df["weighted_defensible"] = weighted_defensible
save_plot_data(defensibility_df, "weighted_defensibility_summary")

display_cols = [
  "model_class",
  "model",
  "average_precision",
  "average_precision_lift_over_random",
  "roc_auc",
]
display(comparison_df[display_cols])
display(defensibility_df)

print("Weighted-model defensibility check")
print(f"best weighted average precision: {best_weighted:.4f}")
print(f"best unweighted average precision: {best_unweighted:.4f}")
if weighted_defensible:
  print(
    "Weighted training is defensible here because its best validation "
    "average precision is best overall or within 2% of the best "
    "unweighted model."
  )
else:
  print(
    "Weighted training is flagged because it trails the best unweighted "
    "model by more than 2% in validation average precision. On a natural "
    "90/10 split, class weighting can improve sensitivity but distort "
    "ranking quality enough that average precision falls."
  )

fig, axes = plt.subplots(
  1,
  2,
  figsize=(14.5, 5.2),
  constrained_layout=True,
)
y_lookup = {label: i for i, label in enumerate(model_order)}
weighted_rows = ["XGBoost weighted", "LightGBM weighted"]
if not weighted_defensible:
  for label in weighted_rows:
    y_value = y_lookup[label]
    axes[0].axhspan(
      y_value - 0.42,
      y_value + 0.42,
      color="red",
      alpha=0.12,
      zorder=0,
    )

colors = {
  "baseline": "0.35",
  "unweighted": "tab:blue",
  "weighted": "tab:orange",
}
for row in comparison_df.itertuples(index=False):
  y_value = y_lookup[row.model_class]
  axes[0].scatter(
    row.average_precision_lift_over_random,
    y_value,
    s=92,
    color=colors[row.training_kind],
    edgecolors="black",
    linewidths=0.8,
    zorder=4,
  )
  axes[0].text(
    row.average_precision_lift_over_random + 0.035,
    y_value,
    f"{row.average_precision_lift_over_random:.2f}x",
    va="center",
    fontsize=8,
  )

axes[0].axvline(
  1.0,
  linestyle="--",
  linewidth=1,
  color="0.25",
  label="Random ranking",
)
for kind, color in colors.items():
  axes[0].scatter(
    [],
    [],
    s=72,
    color=color,
    edgecolors="black",
    linewidths=0.8,
    label=kind.title(),
  )
if not weighted_defensible:
  axes[0].add_patch(Rectangle(
    (0, 0),
    0,
    0,
    color="red",
    alpha=0.12,
    label="Weighted not defensible",
  ))
axes[0].set_yticks(np.arange(len(model_order)))
axes[0].set_yticklabels(model_order)
axes[0].invert_yaxis()
axes[0].set_xlabel("Average precision lift over random")
axes[0].set_title("Validation average precision ranking")
x_values = comparison_df[
  "average_precision_lift_over_random"
].to_numpy()
axes[0].set_xlim(0.85, max(2.20, x_values.max() + 0.45))
axes[0].legend(
  loc="lower left",
  bbox_to_anchor=(0.01, 0.01),
  fontsize=8,
  frameon=True,
)

roc_rows = []
dummy_model = workflow["fitted_models"]["dummy_prior"]
roc_models = [
  ("Dummy prior", dummy_model, "0.35", "--"),
]
optimized_name = workflow.get("optimized_model_name")
best_baseline = results_df[
  results_df["model"].ne(optimized_name)
].iloc[0]["model"]
roc_models.append((
  f"Best baseline before HPO: {best_baseline}",
  workflow["fitted_models"][best_baseline],
  "tab:blue",
  "-",
))
roc_models.append((
  f"Best model after HPO: {workflow['best_model_name']}",
  workflow["best_model"],
  "tab:orange",
  "-",
))

for label, model, color, linestyle in roc_models:
  score = get_score(model, X_valid)
  fpr, tpr, thresholds = roc_curve(y_valid, score)
  axes[1].plot(
    fpr,
    tpr,
    color=color,
    linestyle=linestyle,
    linewidth=2,
    label=label,
  )
  roc_rows.append(pd.DataFrame({
    "label": label,
    "false_positive_rate": fpr,
    "true_positive_rate": tpr,
    "threshold": thresholds,
  }))
axes[1].plot(
  [0, 1],
  [0, 1],
  color="black",
  linestyle=":",
  linewidth=1.5,
  label="Random ROC",
  zorder=5,
)
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].set_title("Validation ROC curves")
axes[1].legend(fontsize=7, loc="lower right", frameon=True)
roc_curves_df = pd.concat(roc_rows, ignore_index=True)
save_plot_data(roc_curves_df, "validation_selected_roc_curves")
save_figure(fig, "validation_model_comparison")
plt.show()


## 13. Calibration metrics and decision matrix


In [ ]:
curve_colors = {
  "precision": "tab:blue",
  "recall": "tab:green",
  "f1": "tab:orange",
  "mcc": "tab:red",
}
curve_labels = {
  "precision": "Precision",
  "recall": "Recall",
  "f1": "F1",
  "mcc": "MCC",
}
threshold_table = workflow["threshold_table"].copy()
save_plot_data(threshold_table, "calibration_threshold_curves")

fig, ax = plt.subplots(figsize=(10.2, 5.6))
for metric, color in curve_colors.items():
  ax.plot(
    threshold_table["threshold"],
    threshold_table[metric],
    color=color,
    linewidth=2.0,
    label=curve_labels[metric],
  )
ax.axvline(
  workflow["max_f1"]["threshold"],
  color="black",
  linestyle="-",
  linewidth=1.8,
  label="Maximum calibration F1",
)
ax.axvline(
  workflow["max_mcc"]["threshold"],
  color="black",
  linestyle="--",
  linewidth=1.8,
  label="Maximum calibration MCC",
)
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Calibration metric")
ax.set_title(
  "Calibration metrics by decision threshold\n"
  f"{workflow['best_model_name']}"
)
ax.grid(linestyle=":", alpha=0.4)
ax.legend(
  loc="center left",
  bbox_to_anchor=(1.02, 0.5),
  fontsize=8,
  frameon=True,
)
fig.tight_layout()
save_figure(fig, "calibration_threshold_metrics")
plt.show()

display(decision_matrix_df)


## 14. Final confusion matrix


In [ ]:
policy = "maximum calibration F1"
prediction = workflow["predictions"][policy]
cm = confusion_matrix(y_test, prediction)
cm_df = pd.DataFrame(
  cm,
  index=["actual_0", "actual_1"],
  columns=["predicted_0", "predicted_1"],
)
cm_export = cm_df.reset_index().rename(
  columns={"index": "actual_class"}
)
save_plot_data(cm_export, "test_confusion_matrix_counts")

fig, ax = plt.subplots(figsize=(5.8, 5.0))
ConfusionMatrixDisplay(cm).plot(
  ax=ax,
  values_format=",d",
  colorbar=False,
)
ax.set_title(
  "Test confusion matrix\n"
  f"{workflow['best_model_name']}\n"
  f"{policy}"
)
fig.tight_layout()
save_figure(fig, "test_confusion_matrix")
plt.show()


## 15. Feature importance

Permutation importance is computed for the selected all-features model 
on the validation split, using average precision as the scoring metric.


In [ ]:
IMPORTANCE_SAMPLE_ROWS = 100_000
IMPORTANCE_REPEATS = 8

importance_n = min(IMPORTANCE_SAMPLE_ROWS, len(X_valid))
X_importance = X_valid.sample(
  importance_n,
  random_state=RANDOM_STATE,
)
y_importance = y_valid.loc[X_importance.index]
perm = permutation_importance(
  workflow["best_model"],
  X_importance,
  y_importance,
  n_repeats=IMPORTANCE_REPEATS,
  scoring="average_precision",
  random_state=RANDOM_STATE,
  n_jobs=1,
)
importance_df = pd.DataFrame({
  "model": workflow["best_model_name"],
  "feature": all_feature_cols,
  "importance_mean": perm.importances_mean,
  "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)
save_plot_data(
  importance_df,
  "validation_permutation_importance",
)
display(importance_df.head(25))

plot_df = importance_df.head(22).iloc[::-1]
fig, ax = plt.subplots(figsize=(8.3, 6.7))
ax.barh(
  plot_df["feature"],
  plot_df["importance_mean"],
  xerr=plot_df["importance_std"],
)
ax.set_xlabel("Decrease in validation average precision")
ax.set_title(
  "Permutation feature importance\n"
  f"{workflow['best_model_name']}"
)
fig.tight_layout()
save_figure(fig, "validation_permutation_importance")
plt.show()


## 16. Leakage and split checks

In [ ]:
forbidden_features = {
  "future_blunder_count",
  "will_blunder_next_move",
  "will_blunder_soon",
  "result",
  "result_white",
  "final_phase_progress",
  "mean_phase_progress",
  "max_phase_progress",
  "final_total_non_pawn_material",
  "final_material_imbalance_white",
}

used_forbidden = forbidden_features.intersection(all_feature_cols)
assert not used_forbidden, used_forbidden

split_groups = {
  "train": set(g_train),
  "valid": set(g_valid),
  "calibration": set(g_calibration),
  "test": set(g_test),
}

split_names = list(split_groups)
for i, left in enumerate(split_names):
  for right in split_names[i + 1:]:
    assert split_groups[left].isdisjoint(split_groups[right])

for split_y in [y_valid, y_calibration, y_test]:
  assert split_y.mean() < 0.5

print("No forbidden feature names are used.")
print("Train/validation/calibration/test games are disjoint.")
print("Evaluation splits retain natural class prevalence.")


## 17. Save selected models and metadata

In [ ]:
saved_model_paths = {}
path = MODELS_DIR / "overall_best.joblib"
joblib.dump(workflow["best_model"], path)
saved_model_paths["overall_best"] = str(path)
print(f"saved: {path}")

metadata = {
  "created_utc": datetime.now(timezone.utc).isoformat(),
  "target": "will_blunder_next_move",
  "horizon_own_moves": HORIZON_OWN_MOVES,
  "blunder_pawn_loss_threshold": BLUNDER_PAWN_LOSS,
  "optimization_metric": "average_precision",
  "best_model_name": workflow["best_model_name"],
  "feature_columns": all_feature_cols,
  "split_summary": split_summary.to_dict(orient="records"),
  "decision_matrix": decision_matrix_df.to_dict(orient="records"),
  "weighted_defensible": bool(weighted_defensible),
  "saved_models": saved_model_paths,
  "figure_directory": str(FIGURES_DIR),
  "plot_data_directory": str(PLOT_DATA_DIR),
}
metadata_path = MODELS_DIR / "model_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(f"saved: {metadata_path}")


## 18. Plot-data workstation


In [ ]:
available_plot_data = sorted(PLOT_DATA_DIR.glob("*.csv"))
print("Exported plot-data files:")
for path in available_plot_data:
  print(path.name)

decision_matrix = pd.read_csv(PLOT_DATA_DIR / "decision_matrix.csv")
display(decision_matrix)
